pull data

In [38]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import pyodbc
# -----------------------------------------------------------------------------
# DATE LOGIC
# -----------------------------------------------------------------------------

today = pd.Timestamp.today().normalize()

# This script is intended to run on a Monday
# Last completed week = Monday -> Sunday immediately before today

start_of_last_week = today - pd.Timedelta(days=today.weekday() + 7)
end_of_last_week = start_of_last_week + pd.Timedelta(days=6)

last_week_dates = pd.date_range(start_of_last_week, end_of_last_week, freq="D")

print(f"Last completed week:")
print(f"Start: {start_of_last_week.date()}")
print(f"End:   {end_of_last_week.date()}")

# -----------------------------------------------------------------------------
# FORECAST FILE LOCATION
# -----------------------------------------------------------------------------

user = Path.home().name

dsn = 'AzureConnection'
# user = 'christina_k'

username = f'{user}@edinburghairport.com'

connect = pyodbc.connect('DSN=' + dsn + ';')

forecast_folder = Path(
    rf"C:\Users\{user}\Edinburgh Airport Limited\Business Planning - Documents\02. Capacity Areas\09. Car Parking\01. FastPark\01 Short-Term Models\Outputs\Daily Forecast Output"
)

# -----------------------------------------------------------------------------
# LOAD ACTUALS
# -----------------------------------------------------------------------------

# Assumes:
# - connect already exists
# - pyodbc/sqlalchemy connection already created

# We need actuals for the last completed week only

sql = f"""
SELECT *
FROM FastPark.v_EntryAndExits
WHERE [CheckInStarted] >= '{start_of_last_week.strftime('%Y-%m-%d')}'
    AND [CheckInStarted] < '{(end_of_last_week + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
"""

fastpark_actual_entries_df = pd.read_sql(sql, connect)

fastpark_actual_entries_df = fastpark_actual_entries_df.drop_duplicates(subset='BookingReference')


sql = f"""
SELECT *
FROM FastPark.v_EntryAndExits
WHERE [ActualCheckedOutDate] >= '{start_of_last_week.strftime('%Y-%m-%d')}'
    AND [ActualCheckedOutDate] < '{(end_of_last_week + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
"""

fastpark_actual_exits_df = pd.read_sql(sql, connect)

fastpark_actual_exits_df = fastpark_actual_exits_df.drop_duplicates(subset='BookingReference')

# -----------------------------------------------------------------------------
# PREP ACTUALS
# -----------------------------------------------------------------------------

# Entries
fastpark_actual_entries_df["Date"] = pd.to_datetime(
    fastpark_actual_entries_df["CheckInStarted"]
).dt.date

actual_entries_daily = (
    fastpark_actual_entries_df
    .groupby("Date")
    .size()
    .reset_index(name="ActualEntries")
)

# Exits
fastpark_actual_exits_df["Date"] = pd.to_datetime(
    fastpark_actual_exits_df["ActualCheckedOutDate"]
).dt.date

actual_exits_daily = (
    fastpark_actual_exits_df
    .groupby("Date")
    .size()
    .reset_index(name="ActualExits")
)

# Merge actuals
actuals_daily = pd.merge(
    actual_entries_daily,
    actual_exits_daily,
    on="Date",
    how="outer"
).fillna(0)

actuals_daily["Date"] = pd.to_datetime(actuals_daily["Date"])

print("\nActuals daily:")
print(actuals_daily.head())

# -----------------------------------------------------------------------------
# LEAD TIMES TO CHECK
# -----------------------------------------------------------------------------

lead_times = [21, 14, 7, 3, 1]

# Results dictionary
entries_accuracy_results = {}
exits_accuracy_results = {}
# Results dictionary
entries_accuracy_results = {}
exits_accuracy_results = {}

# NEW
entries_forecast_results = {}
exits_forecast_results = {}

entries_actual_results = {}
exits_actual_results = {}
# -----------------------------------------------------------------------------
# LOOP THROUGH EACH DAY IN LAST WEEK
# -----------------------------------------------------------------------------

for target_date in last_week_dates:

    target_date_only = target_date.date()

    # Get actuals
    actual_row = actuals_daily[
        actuals_daily["Date"].dt.date == target_date_only
    ]

    if actual_row.empty:
        print(f"\nNo actuals found for {target_date_only}")
        continue

    actual_entries = actual_row["ActualEntries"].iloc[0]
    actual_exits = actual_row["ActualExits"].iloc[0]

    entries_accuracy_results[target_date_only] = {}
    exits_accuracy_results[target_date_only] = {}
    
    entries_forecast_results[target_date_only] = {}
    exits_forecast_results[target_date_only] = {}

    entries_actual_results[target_date_only] = {}
    exits_actual_results[target_date_only] = {}
    # -------------------------------------------------------------------------
    # CHECK EACH LEAD TIME
    # -------------------------------------------------------------------------

    for lead in lead_times:

        # Always store actuals for every lead so gaps in forecast files
        # do not create blank actuals across horizon columns.
        entries_actual_results[target_date_only][lead] = actual_entries
        exits_actual_results[target_date_only][lead] = actual_exits

        # Forecast file date
        forecast_run_date = target_date - pd.Timedelta(days=lead)

        if forecast_run_date.strftime('%Y-%m-%d') == '2026-06-22':
            forecast_filename = (
                f"{forecast_run_date.strftime('%Y-%m-%d')}_daily_forecast - Copy.csv"
            )
        else:
            forecast_filename = (
                f"{forecast_run_date.strftime('%Y-%m-%d')}_daily_forecast.csv"
            )

        forecast_path = forecast_folder / forecast_filename

        print(f"\nChecking forecast file:")
        print(f"Target date: {target_date_only}")
        print(f"Lead time: {lead}")
        print(f"Forecast file: {forecast_filename}")

        # ---------------------------------------------------------------------
        # FILE EXISTS?
        # ---------------------------------------------------------------------

        if not forecast_path.exists():
            print("Forecast file NOT found - skipping")
            entries_accuracy_results[target_date_only][lead] = np.nan
            exits_accuracy_results[target_date_only][lead] = np.nan
            continue

        try:

            # -----------------------------------------------------------------
            # LOAD FORECAST FILE
            # -----------------------------------------------------------------
            # -----------------------------------------------------------------
            # LOAD FORECAST FILE
            # -----------------------------------------------------------------

            forecast_df = pd.read_csv(forecast_path)

            print(f"Loaded rows: {len(forecast_df)}")

            # -----------------------------------------------------------------
            # HANDLE POSSIBLE MISSING / DIFFERENT COLUMN NAMES
            # -----------------------------------------------------------------

            # Strip spaces from column names first
            forecast_df.columns = [str(c).strip() for c in forecast_df.columns]

            print("Original columns:")
            print(forecast_df.columns.tolist())

            # -------------------------------------------------------------
            # HANDLE DATE COLUMN
            # -------------------------------------------------------------

            if "Date" not in forecast_df.columns:

                print("Date column header missing - renaming first column to 'Date'")

                cols = list(forecast_df.columns)
                cols[0] = "Date"

                forecast_df.columns = cols

            # -------------------------------------------------------------
            # HANDLE ENTRIES COLUMN
            # -------------------------------------------------------------

            entries_col = None

            possible_entries_cols = [
                "Entries",
                "entry_forecast"
            ]

            for col in possible_entries_cols:
                if col in forecast_df.columns:
                    entries_col = col
                    break

            if entries_col is None:
                print("No entries forecast column found")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            # -------------------------------------------------------------
            # HANDLE EXITS COLUMN
            # -------------------------------------------------------------

            exits_col = None

            possible_exits_cols = [
                "Exits",
                "exit_forecast"
            ]

            for col in possible_exits_cols:
                if col in forecast_df.columns:
                    exits_col = col
                    break

            if exits_col is None:
                print("No exits forecast column found")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            print(f"Using entries column: {entries_col}")
            print(f"Using exits column:   {exits_col}")

            # -----------------------------------------------------------------
            # CLEAN DATE
            # -----------------------------------------------------------------
            date_sample = str(forecast_df["Date"].dropna().iloc[0])

            print(f"Sample date value: {date_sample}")

            # Detect format automatically
            if "-" in date_sample:
                
                # Likely YYYY-MM-DD
                forecast_df["Date"] = pd.to_datetime(
                    forecast_df["Date"],
                    format="%Y-%m-%d",
                    errors="coerce"
                )

            else:
                
                # Likely DD/MM/YYYY
                forecast_df["Date"] = pd.to_datetime(
                    forecast_df["Date"],
                    format="%d/%m/%Y",
                    errors="coerce"
                )

            # Debugging
            if forecast_df["Date"].isna().all():
                print("All forecast dates failed to parse")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue
            # -----------------------------------------------------------------
            # FILTER TO TARGET DATE
            # -----------------------------------------------------------------

            target_forecast = forecast_df[
                forecast_df["Date"].dt.date == target_date_only
            ]

            if target_forecast.empty:
                print("No matching forecast rows for target date")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            # -----------------------------------------------------------------
            # DAILY FORECAST TOTALS
            # -----------------------------------------------------------------

            forecast_entries = pd.to_numeric(
                target_forecast[entries_col],
                errors="coerce"
            ).fillna(0).sum()

            forecast_exits = pd.to_numeric(
                target_forecast[exits_col],
                errors="coerce"
            ).fillna(0).sum()

            print(f"Forecast Entries: {forecast_entries:.2f}")
            print(f"Forecast Exits:   {forecast_exits:.2f}")

            print(f"Actual Entries:   {actual_entries}")
            print(f"Actual Exits:     {actual_exits}")

            # Store forecasts
            entries_forecast_results[target_date_only][lead] = forecast_entries
            exits_forecast_results[target_date_only][lead] = forecast_exits

            # Store actuals
            entries_actual_results[target_date_only][lead] = actual_entries
            exits_actual_results[target_date_only][lead] = actual_exits


            # -----------------------------------------------------------------
            # ACCURACY CALC
            # Formula:
            # (Actual - Forecast) / Actual
            # -----------------------------------------------------------------

            if actual_entries != 0:
                entries_accuracy = 1-abs((
                    (actual_entries - forecast_entries)
                    / actual_entries
                ))
            else:
                entries_accuracy = np.nan

            if actual_exits != 0:
                exits_accuracy = 1-abs((
                    (actual_exits - forecast_exits)
                    / actual_exits
                ))
            else:
                exits_accuracy = np.nan

            entries_accuracy_results[target_date_only][lead] = entries_accuracy
            exits_accuracy_results[target_date_only][lead] = exits_accuracy

            print(f"Entries Accuracy: {entries_accuracy:.4f}")
            print(f"Exits Accuracy:   {exits_accuracy:.4f}")

        except Exception as e:

            print(f"ERROR reading forecast file:")
            print(f"{forecast_filename}")
            print(str(e))

            entries_accuracy_results[target_date_only][lead] = np.nan
            exits_accuracy_results[target_date_only][lead] = np.nan

# -----------------------------------------------------------------------------
# FINAL DATAFRAMES
# -----------------------------------------------------------------------------

entries_accuracy_df = pd.DataFrame.from_dict(
    entries_accuracy_results,
    orient="index"
)

exits_accuracy_df = pd.DataFrame.from_dict(
    exits_accuracy_results,
    orient="index"
)

entries_accuracy_df.index.name = "Date"
exits_accuracy_df.index.name = "Date"

entries_accuracy_df = entries_accuracy_df.sort_index()
exits_accuracy_df = exits_accuracy_df.sort_index()

# Rename columns
entries_accuracy_df.columns = [f"{c}_days_out" for c in entries_accuracy_df.columns]
exits_accuracy_df.columns = [f"{c}_days_out" for c in exits_accuracy_df.columns]

# -----------------------------------------------------------------------------
# OUTPUT
# -----------------------------------------------------------------------------

print("\n================ ENTRIES ACCURACY ================\n")
print(entries_accuracy_df)

print("\n================ EXITS ACCURACY ================\n")
print(exits_accuracy_df)

# -----------------------------------------------------------------------------
# FORECAST VALUE TABLES
# -----------------------------------------------------------------------------

entries_forecast_df = pd.DataFrame.from_dict(
    entries_forecast_results,
    orient="index"
)

exits_forecast_df = pd.DataFrame.from_dict(
    exits_forecast_results,
    orient="index"
)

entries_forecast_df.index.name = "Date"
exits_forecast_df.index.name = "Date"

entries_forecast_df = entries_forecast_df.sort_index()
exits_forecast_df = exits_forecast_df.sort_index()

entries_forecast_df.columns = [
    f"{c}_days_out" for c in entries_forecast_df.columns
]

exits_forecast_df.columns = [
    f"{c}_days_out" for c in exits_forecast_df.columns
]

# -----------------------------------------------------------------------------
# ACTUAL VALUE TABLES
# -----------------------------------------------------------------------------

entries_actual_df = pd.DataFrame.from_dict(
    entries_actual_results,
    orient="index"
)

exits_actual_df = pd.DataFrame.from_dict(
    exits_actual_results,
    orient="index"
)

entries_actual_df.index.name = "Date"
exits_actual_df.index.name = "Date"

entries_actual_df = entries_actual_df.sort_index()
exits_actual_df = exits_actual_df.sort_index()

# Fill across horizon columns for the same date so actuals remain consistent
# even when some forecast horizons are missing.
entries_actual_df = entries_actual_df.bfill(axis=1).ffill(axis=1)
exits_actual_df = exits_actual_df.bfill(axis=1).ffill(axis=1)

entries_actual_df.columns = [
    f"{c}_days_out" for c in entries_actual_df.columns
]

exits_actual_df.columns = [
    f"{c}_days_out" for c in exits_actual_df.columns
]

# -----------------------------------------------------------------------------
# PRINT OUTPUTS
# -----------------------------------------------------------------------------

print("\n================ ENTRIES FORECASTS ================\n")
print(entries_forecast_df)

print("\n================ EXITS FORECASTS ================\n")
print(exits_forecast_df)

print("\n================ ENTRIES ACTUALS ================\n")
print(entries_actual_df)

print("\n================ EXITS ACTUALS ================\n")
print(exits_actual_df)

Last completed week:
Start: 2026-10-19
End:   2026-10-25


C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_25948\1953847077.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fastpark_actual_entries_df = pd.read_sql(sql, connect)
C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_25948\1953847077.py:70: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fastpark_actual_exits_df = pd.read_sql(sql, connect)



Actuals daily:
        Date  ActualEntries  ActualExits
0 2026-10-19            546          775
1 2026-10-20            409          759
2 2026-10-21            489          589
3 2026-10-22            439          776
4 2026-10-23            515          658

Checking forecast file:
Target date: 2026-10-19
Lead time: 21
Forecast file: 2026-09-28_daily_forecast.csv
Loaded rows: 60
Original columns:
['Unnamed: 0', 'booking_entries', 'booking_exits', 'entry_forecast', 'exit_forecast']
Date column header missing - renaming first column to 'Date'
Using entries column: entry_forecast
Using exits column:   exit_forecast
Sample date value: 2026-09-29
Forecast Entries: 763.00
Forecast Exits:   755.00
Actual Entries:   546
Actual Exits:     775
Entries Accuracy: 0.6026
Exits Accuracy:   0.9742

Checking forecast file:
Target date: 2026-10-19
Lead time: 14
Forecast file: 2026-10-05_daily_forecast.csv
Loaded rows: 60
Original columns:
['Unnamed: 0', 'booking_entries', 'booking_exits', 'entry_fo

save to excel

In [39]:
# -----------------------------------------------------------------------------
# SAVE TO EXCEL
# -----------------------------------------------------------------------------

output_folder = Path(
    rf"C:\Users\{user}\Edinburgh Airport Limited\Business Planning - Documents\02. Capacity Areas\09. Car Parking\01. FastPark\01 Short-Term Models\FastPark Short Term Model Oct 2026\06 Comparison Framework\Forecast accuracy"

)

output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / (
    f"forecast_accuracy_{pd.Timestamp.today().strftime('%d%m%y')}.xlsx"
)

# -----------------------------------------------------------------------------
# WRITE TO EXCEL
# -----------------------------------------------------------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    sheet_name = "Forecast Accuracy"

    # -------------------------------------------------------------------------
    # ENTRIES SECTION
    # -------------------------------------------------------------------------

    current_row = 0

    # -------------------------
    # ENTRIES FORECAST
    # -------------------------

    entries_forecast_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet = writer.sheets[sheet_name]
    worksheet.cell(row=current_row + 1, column=1).value = "Entries Forecast"

    current_row += len(entries_forecast_df) + 4

    # -------------------------
    # ENTRIES ACTUALS
    # -------------------------

    entries_actual_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet.cell(row=current_row + 1, column=1).value = "Entries Actuals"

    current_row += len(entries_actual_df) + 4

    # -------------------------
    # ENTRIES ACCURACY
    # -------------------------

    entries_accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet.cell(row=current_row + 1, column=1).value = "Entries Accuracy"

    # -------------------------------------------------------------------------
    # EXITS SECTION
    # -------------------------------------------------------------------------

    exits_start_col = len(entries_forecast_df.columns) + 4

    current_row = 0

    # -------------------------
    # EXITS FORECAST
    # -------------------------

    exits_forecast_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Forecast"

    current_row += len(exits_forecast_df) + 4

    # -------------------------
    # EXITS ACTUALS
    # -------------------------

    exits_actual_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Actuals"

    current_row += len(exits_actual_df) + 4

    # -------------------------
    # EXITS ACCURACY
    # -------------------------

    exits_accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Accuracy"

    # -------------------------------------------------------------------------
    # FORMAT ACCURACY TABLES AS PERCENTAGES
    # -------------------------------------------------------------------------

    from openpyxl.styles import Font

    bold_font = Font(bold=True)

    # Bold all titles
    for row in worksheet.iter_rows():
        for cell in row:
            if cell.value in [
                "Entries Forecast",
                "Entries Actuals",
                "Entries Accuracy",
                "Exits Forecast",
                "Exits Actuals",
                "Exits Accuracy"
            ]:
                cell.font = bold_font

    # Auto-width columns
    for column_cells in worksheet.columns:

        length = 0
        column = column_cells[0].column_letter

        for cell in column_cells:
            try:
                length = max(length, len(str(cell.value)))
            except:
                pass

        worksheet.column_dimensions[column].width = length + 3

print(f"\nExcel saved:")
print(output_file)


Excel saved:
C:\Users\jamie_douglas\Edinburgh Airport Limited\Business Planning - Documents\02. Capacity Areas\09. Car Parking\01. FastPark\01 Short-Term Models\FastPark Short Term Model Oct 2026\06 Comparison Framework\Forecast accuracy\forecast_accuracy_301026.xlsx


plot accuracy lines

In [40]:
import plotly.graph_objects as go

# -----------------------------------------------------------------------------
# PREP DATA
# -----------------------------------------------------------------------------

# Create formatted x-axis labels
entries_plot_df = entries_accuracy_df.copy()
exits_plot_df = exits_accuracy_df.copy()

entries_plot_df.index = pd.to_datetime(entries_plot_df.index)
exits_plot_df.index = pd.to_datetime(exits_plot_df.index)

x_labels = entries_plot_df.index.strftime("%a %d-%m")

# -----------------------------------------------------------------------------
# ENTRIES ACCURACY CHART
# -----------------------------------------------------------------------------

fig_entries = go.Figure()

for col in ["21_days_out", "7_days_out", "3_days_out"]:

    if col in entries_plot_df.columns:

        fig_entries.add_trace(
            go.Scatter(
                x=x_labels,
                y=entries_plot_df[col] * 100,
                mode="lines+markers",
                name=col.replace("_days_out", " Days Out")
            )
        )

fig_entries.update_layout(
    title="FastPark Entries Forecast Accuracy",
    xaxis_title="Day",
    yaxis_title="Accuracy %",
    yaxis=dict(
        ticksuffix="%",
        range=[0, 100]
    ),
    hovermode="x unified",
    template="plotly_white"
)

fig_entries.show()

# -----------------------------------------------------------------------------
# EXITS ACCURACY CHART
# -----------------------------------------------------------------------------

fig_exits = go.Figure()

for col in ["21_days_out", "7_days_out", "3_days_out"]:

    if col in exits_plot_df.columns:

        fig_exits.add_trace(
            go.Scatter(
                x=x_labels,
                y=exits_plot_df[col] * 100,
                mode="lines+markers",
                name=col.replace("_days_out", " Days Out")
            )
        )

fig_exits.update_layout(
    title="FastPark Exits Forecast Accuracy",
    xaxis_title="Day",
    yaxis_title="Accuracy %",
    yaxis=dict(
        ticksuffix="%",
        range=[0, 100]
    ),
    hovermode="x unified",
    template="plotly_white"
)

fig_exits.show()

plot actuals vs forecast

In [41]:
import plotly.graph_objects as go
import pandas as pd

# -----------------------------------------------------------------------------
# PREP DATA
# -----------------------------------------------------------------------------

entries_plot_df = entries_accuracy_df.copy()
exits_plot_df = exits_accuracy_df.copy()

entries_forecast_plot = entries_forecast_df.copy()
exits_forecast_plot = exits_forecast_df.copy()

entries_actual_plot = entries_actual_df.copy()
exits_actual_plot = exits_actual_df.copy()

entries_plot_df.index = pd.to_datetime(entries_plot_df.index)
exits_plot_df.index = pd.to_datetime(exits_plot_df.index)

entries_forecast_plot.index = pd.to_datetime(entries_forecast_plot.index)
exits_forecast_plot.index = pd.to_datetime(exits_forecast_plot.index)

entries_actual_plot.index = pd.to_datetime(entries_actual_plot.index)
exits_actual_plot.index = pd.to_datetime(exits_actual_plot.index)

x_labels = entries_plot_df.index.strftime("%a %d-%m")

lead_cols = ["21_days_out", "7_days_out", "3_days_out"]

# Colour mapping
colors = {
    "21_days_out": "#ef9a00",
    "7_days_out": "#3386a8",
    "3_days_out": "#ce007f"
}
# =============================================================================
# ENTRIES FIGURE
# =============================================================================

fig_entries = go.Figure()

actual_entries = entries_actual_plot.iloc[:, 0]

# -----------------------------------------------------------------------------
# ACTUALS (GREY BARS)
# -----------------------------------------------------------------------------

fig_entries.add_trace(
    go.Bar(
        x=x_labels,
        y=actual_entries.values,
        name="Actual Entries",
        marker=dict(color="lightgrey"),
        text=actual_entries.values,
        textposition="inside",
        insidetextanchor="middle",
        textangle=90,
        textfont=dict(size=16, color="black"),
        hovertemplate="Actual: %{y}<extra></extra>"
    )
)

# -----------------------------------------------------------------------------
# FORECAST LINES + ACCURACY LABELS
# -----------------------------------------------------------------------------

for col in lead_cols:

    if col in entries_forecast_plot.columns and col in entries_plot_df.columns:

        line_colour = colors.get(col, "black")

        fig_entries.add_trace(
            go.Scatter(
                x=x_labels,
                y=entries_forecast_plot[col].values,
                mode="lines+markers+text",
                name=f"Forecast {col.replace('_days_out','d out')}",
                line=dict(color=line_colour),
                marker=dict(color=line_colour),
                text=(entries_plot_df[col] * 100).round(0).map(lambda v: "" if pd.isna(v) else f"{int(v)}%"),
                textposition="top center",
                textfont=dict(color=line_colour),
                hovertemplate="Forecast: %{y}<extra></extra>"
            )
        )

# -----------------------------------------------------------------------------
# LAYOUT
# -----------------------------------------------------------------------------

fig_entries.update_layout(
    title="FastPark Entries Actuals vs Forecast",
    xaxis_title="Day",
    yaxis_title="Entries",
    barmode="group",
    template="plotly_white"
)

fig_entries.show()

# =============================================================================
# EXITS FIGURE
# =============================================================================

fig_exits = go.Figure()

actual_exits = exits_actual_plot.iloc[:, 0]

# -----------------------------------------------------------------------------
# ACTUALS (GREY BARS)
# -----------------------------------------------------------------------------

fig_exits.add_trace(
    go.Bar(
        x=x_labels,
        y=actual_exits.values,
        name="Actual Exits",
        marker=dict(color="lightgrey"),
        text=actual_exits.values,
        textposition="inside",
        insidetextanchor="middle",
        textangle=90,
        textfont=dict(size=16, color="black"),
        hovertemplate="Actual: %{y}<extra></extra>"
    )
)

# -----------------------------------------------------------------------------
# FORECAST LINES + ACCURACY LABELS
# -----------------------------------------------------------------------------

for col in lead_cols:

    if col in exits_forecast_plot.columns and col in exits_plot_df.columns:

        line_colour = colors.get(col, "black")

        fig_exits.add_trace(
            go.Scatter(
                x=x_labels,
                y=exits_forecast_plot[col].values,
                mode="lines+markers+text",
                name=f"Forecast {col.replace('_days_out','d out')}",
                line=dict(color=line_colour),
                marker=dict(color=line_colour),
                text=(exits_plot_df[col] * 100).round(0).map(lambda v: "" if pd.isna(v) else f"{int(v)}%"),
                textposition="top center",
                textfont=dict(color=line_colour),
                hovertemplate="Forecast: %{y}<extra></extra>"
            )
        )

fig_exits.update_layout(
    title="FastPark Exits Actuals vs Forecast",
    xaxis_title="Day",
    yaxis_title="Exits",
    barmode="group",
    template="plotly_white"
)

fig_exits.show()